In [0]:
%sql
select COD_CIRCUITO_MT, ANO, MES, sum(EXTENSAO)/1000 as KM from poseidon_uc.eospraw.total_ext_rede_mt group by COD_CIRCUITO_MT, ANO, MES order by ANO, MES, COD_CIRCUITO_MT

In [0]:
%sql

DECLARE OR REPLACE VARIABLE p_workspace_id STRING DEFAULT '8568530819532784';
DECLARE OR REPLACE VARIABLE p_owner STRING DEFAULT 'jader.campos@edpbr.com.br';

DECLARE OR REPLACE VARIABLE p_start_date DATE DEFAULT DATE_SUB(CURRENT_DATE(), 90);
DECLARE OR REPLACE VARIABLE p_end_date DATE DEFAULT CURRENT_DATE();

WITH job_info AS (
    SELECT
        workspace_id,
        job_id,
        name AS job_name,
        creator_user_name AS job_owner
    FROM (
        SELECT
            workspace_id,
            job_id,
            name,
            creator_user_name,
            ROW_NUMBER() OVER (
                PARTITION BY workspace_id, job_id
                ORDER BY change_time DESC
            ) AS rn
        FROM system.lakeflow.jobs
        WHERE workspace_id = p_workspace_id
          AND creator_user_name = p_owner
    )
    WHERE rn = 1
),

usage_with_price AS (
    SELECT
        u.workspace_id,
        u.usage_date,
        u.sku_name,
        u.usage_quantity,
        u.usage_metadata.job_id AS job_id,
        u.usage_metadata.job_run_id AS job_run_id,
        p.pricing.default AS price_per_unit
    FROM system.billing.usage u
    LEFT JOIN system.billing.list_prices p
        ON u.sku_name = p.sku_name
       AND u.usage_unit = p.usage_unit
       AND u.usage_start_time >= p.price_start_time
       AND (u.usage_start_time < p.price_end_time OR p.price_end_time IS NULL)
    WHERE u.workspace_id = p_workspace_id
      AND u.usage_metadata.job_id IS NOT NULL
      AND u.usage_date BETWEEN p_start_date AND p_end_date
),

run_cost AS (
    SELECT
        j.job_id,
        j.job_name,
        j.job_owner,
        up.job_run_id,
        up.usage_date,
        CASE
            WHEN up.sku_name LIKE '%ALL_PURPOSE%' THEN 'All-Purpose Compute'
            WHEN up.sku_name LIKE '%JOBS_SERVERLESS%' THEN 'Serverless Jobs'
            WHEN up.sku_name LIKE '%JOBS%' THEN 'Job Compute'
            ELSE up.sku_name
        END AS compute_type,
        SUM(up.usage_quantity * up.price_per_unit) AS run_cost_usd
    FROM job_info j
    INNER JOIN usage_with_price up
        ON j.job_id = up.job_id
       AND j.workspace_id = up.workspace_id
    GROUP BY j.job_id, j.job_name, j.job_owner, up.job_run_id, up.usage_date, compute_type
),

pivoted AS (
    SELECT
        job_id,
        job_name,
        job_owner,
        job_run_id,
        usage_date,
        SUM(CASE WHEN compute_type = 'All-Purpose Compute' THEN run_cost_usd ELSE 0 END) AS cost_all_purpose,
        SUM(CASE WHEN compute_type = 'Job Compute' THEN run_cost_usd ELSE 0 END) AS cost_job_compute,
        SUM(CASE WHEN compute_type = 'Serverless Jobs' THEN run_cost_usd ELSE 0 END) AS cost_serverless,
        SUM(run_cost_usd) AS total_cost_run
    FROM run_cost
    GROUP BY job_id, job_name, job_owner, job_run_id, usage_date
),

ranked AS (
    SELECT
        *,
        CASE
            WHEN cost_all_purpose >= cost_job_compute AND cost_all_purpose >= cost_serverless THEN 'All-Purpose Compute'
            WHEN cost_job_compute >= cost_all_purpose AND cost_job_compute >= cost_serverless THEN 'Job Compute'
            ELSE 'Serverless Jobs'
        END AS predominant_compute
    FROM pivoted
),

run_duration AS (
    SELECT
        job_id,
        run_id AS job_run_id,
        MIN(period_start_time) AS run_start_time,
        MAX(period_end_time) AS run_end_time,
        SUM(
            CASE WHEN period_end_time IS NOT NULL
            THEN (UNIX_TIMESTAMP(period_end_time) - UNIX_TIMESTAMP(period_start_time)) / 60.0
            ELSE 0 END
        ) AS duration_minutes
    FROM system.lakeflow.job_run_timeline
    WHERE workspace_id = p_workspace_id
    GROUP BY job_id, run_id
)

SELECT
    r.job_id,
    r.job_name,
    r.job_owner,
    r.job_run_id,
    r.usage_date,
    d.run_start_time,
    d.run_end_time,
    d.duration_minutes,
    r.cost_all_purpose,
    r.cost_job_compute,
    r.cost_serverless,
    r.total_cost_run,
    CASE
        WHEN d.duration_minutes > 0 THEN ROUND(r.total_cost_run / d.duration_minutes, 4)
        ELSE NULL
    END AS cost_per_minute,
    r.predominant_compute,
    LAG(r.predominant_compute) OVER (PARTITION BY r.job_id ORDER BY r.usage_date, r.job_run_id) AS previous_predominant_compute,
    CASE
        WHEN r.predominant_compute != LAG(r.predominant_compute) OVER (PARTITION BY r.job_id ORDER BY r.usage_date, r.job_run_id)
        THEN TRUE
        ELSE FALSE
    END AS compute_type_changed
FROM ranked r
LEFT JOIN run_duration d
    ON r.job_id = d.job_id
   AND r.job_run_id = d.job_run_id
ORDER BY r.job_id, r.usage_date, r.job_run_id;